# 阶段二：Colab T4 QLoRA SFT

从 Colab 菜单选择 **Runtime → Change runtime type**：Runtime Version 选择 `2026.04`，Hardware accelerator 选择单张 `T4 GPU`。然后按顺序运行全部单元。任一断言失败都应停止；不要自动换版本、换模型或调参。

本 notebook 会在 `/content` 训练，并把每次运行写入 Drive 的独立 UTC `run-id` 目录。基座权重只留在 Colab/Hugging Face 缓存中。

In [ ]:
# 1. 在安装任何依赖前验证 Colab 2026.04 的关键指纹和单张 T4。
import json
import os
import platform
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
except ImportError as exc:
    raise RuntimeError("必须在 Google Colab 托管运行时执行") from exc

import torch

assert sys.version_info[:2] == (3, 12), platform.python_version()
assert torch.__version__.split("+")[0] == "2.10.0", torch.__version__
assert torch.cuda.is_available(), "CUDA 不可用"
assert torch.cuda.device_count() == 1, torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
assert "T4" in gpu_name, gpu_name
assert gpu_memory_gib >= 14.0, gpu_memory_gib
nvidia_smi = subprocess.run(
    ["nvidia-smi"], check=True, text=True, capture_output=True
).stdout
runtime_preflight = {
    "required_colab_runtime": "2026.04",
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": gpu_name,
    "gpu_memory_gib": round(gpu_memory_gib, 2),
}
print(json.dumps(runtime_preflight, ensure_ascii=False, indent=2))
print(nvidia_smi)

In [ ]:
# 2. 挂载 Drive，从公开 main 克隆仓库，并记录实际 commit 与独立 run-id。
from datetime import datetime, timezone
from uuid import uuid4

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/chenkx612/local-roleplay-llm.git"
REPO_DIR = Path("/content/local-roleplay-llm")
assert not REPO_DIR.exists(), f"为避免使用旧 checkout，请重启 runtime：{REPO_DIR}"
subprocess.run(
    ["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)],
    check=True,
)
repo_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, check=True, text=True, capture_output=True
).stdout.strip()

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
RUN_ROOT = Path("/content/roleplay-stage2-runs") / RUN_ID
DRIVE_RUN_ROOT = (
    Path("/content/drive/MyDrive/roleplay/morgana-v1/stage2-sft") / RUN_ID
)
RUN_ROOT.mkdir(parents=True, exist_ok=False)
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=False)
CONFIG_PATH = REPO_DIR / "configs/morgana_v1_sft_t4.yaml"
NOTEBOOK_PATH = REPO_DIR / "notebooks/stage2_sft_colab.ipynb"
assert CONFIG_PATH.is_file() and NOTEBOOK_PATH.is_file(), (CONFIG_PATH, NOTEBOOK_PATH)
os.chdir(REPO_DIR)
run_context = {
    "run_id": RUN_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "repository": REPO_URL,
    "branch": "main",
    "commit": repo_commit,
    "temporary_root": str(RUN_ROOT),
    "drive_root": str(DRIVE_RUN_ROOT),
}
(RUN_ROOT / "run_context.json").write_text(
    json.dumps(run_context, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(run_context, ensure_ascii=False, indent=2))

## 固定依赖

安装失败或版本不完全一致时停止。本阶段不安装 FlashAttention、DeepSpeed、vLLM 或在线实验追踪服务。普通依赖从 PyPI wheel 安装；`causal-conv1d` 使用上游为 Python 3.12、CUDA 12、PyTorch 2.10 和 CXX11 ABI 发布的预编译 wheel，避免在 Colab T4 上从源码构建 CUDA 扩展。

In [ ]:
# 3. 严格安装并核验依赖。
import importlib.metadata
import shutil

from packaging.markers import default_environment
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name

PINNED_PACKAGES = {
    "ms-swift": "4.4.1",
    "transformers": "5.12.1",
    "peft": "0.19.1",
    "bitsandbytes": "0.49.2",
    "qwen-vl-utils": "0.0.14",
    "flash-linear-attention": "0.5.1",
    "ninja": "1.13.0",
    "causal-conv1d": "1.6.2.post1",
}
assert torch.version.cuda and torch.version.cuda.startswith("12."), torch.version.cuda
assert torch._C._GLIBCXX_USE_CXX11_ABI is True, torch._C._GLIBCXX_USE_CXX11_ABI

python_dependencies_command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    *[
        f"{name}=={version}"
        for name, version in PINNED_PACKAGES.items()
        if name != "causal-conv1d"
    ],
]
subprocess.run(python_dependencies_command, check=True)

causal_version = PINNED_PACKAGES["causal-conv1d"]
causal_wheel_url = (
    "https://github.com/Dao-AILab/causal-conv1d/releases/download/"
    f"v{causal_version}/causal_conv1d-{causal_version}%2B"
    "cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "--no-deps", causal_wheel_url],
    check=True,
)
installed_versions = {name: importlib.metadata.version(name) for name in PINNED_PACKAGES}
normalized_versions = {name: version.split("+", 1)[0] for name, version in installed_versions.items()}
assert normalized_versions == PINNED_PACKAGES, installed_versions

def validate_dependency_closure(root_names):
    environment = default_environment()
    environment["extra"] = ""
    pending = [canonicalize_name(name) for name in root_names]
    validated = set()
    issues = []
    while pending:
        name = pending.pop()
        if name in validated:
            continue
        validated.add(name)
        try:
            distribution = importlib.metadata.distribution(name)
        except importlib.metadata.PackageNotFoundError:
            issues.append(f"缺少依赖: {name}")
            continue
        for requirement_text in distribution.requires or []:
            requirement = Requirement(requirement_text)
            if requirement.marker and not requirement.marker.evaluate(environment):
                continue
            dependency = canonicalize_name(requirement.name)
            try:
                dependency_version = importlib.metadata.version(dependency)
            except importlib.metadata.PackageNotFoundError:
                issues.append(f"{name} 要求 {requirement}，但未安装")
                continue
            if requirement.specifier and not requirement.specifier.contains(
                dependency_version, prereleases=True
            ):
                issues.append(
                    f"{name} 要求 {requirement}，实际为 {dependency_version}"
                )
            pending.append(dependency)
    return sorted(set(issues)), sorted(validated)

dependency_issues, validated_distributions = validate_dependency_closure(PINNED_PACKAGES)
assert not dependency_issues, "训练依赖闭包不一致:\n" + "\n".join(dependency_issues)
pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    check=False,
    text=True,
    capture_output=True,
)
pip_check_output = "\n".join(
    part.strip() for part in (pip_check.stdout, pip_check.stderr) if part.strip()
)
if pip_check.returncode:
    print("Colab 全局 pip check 报告（仅记录；训练依赖闭包已单独通过）：")
    print(pip_check_output)

def sync_to_drive(source: Path, relative_name: str | None = None) -> Path:
    destination = DRIVE_RUN_ROOT / (relative_name or source.name)
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
    return destination

environment_dir = RUN_ROOT / "environment"
environment_dir.mkdir()
environment_manifest = {
    **runtime_preflight,
    "repository_commit": repo_commit,
    "packages": installed_versions,
    "dependency_validation": {
        "scope": "pinned training dependency closure",
        "validated_distributions": validated_distributions,
        "issues": dependency_issues,
    },
    "global_pip_check_returncode": pip_check.returncode,
}
(environment_dir / "environment.json").write_text(
    json.dumps(environment_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
(environment_dir / "nvidia-smi.txt").write_text(nvidia_smi, encoding="utf-8")
(environment_dir / "pip-check.txt").write_text(
    f"returncode={pip_check.returncode}\n{pip_check_output}\n", encoding="utf-8"
)
pip_freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"], check=True, text=True, capture_output=True
).stdout
(environment_dir / "pip-freeze.txt").write_text(pip_freeze, encoding="utf-8")
sync_to_drive(environment_dir, "environment")
sync_to_drive(RUN_ROOT / "run_context.json")
print(json.dumps(installed_versions, ensure_ascii=False, indent=2))

In [ ]:
# 4. 冻结输入、结构、模型 revision 与训练模板 token 长度校验。
import hashlib

from huggingface_hub import HfApi
from transformers import AutoProcessor
import yaml

MODEL_ID = "Qwen/Qwen3.5-2B"
MODEL_REVISION = "965dcc54bc9c0591873df0e9869c056a54d323d1"
EXPECTED_INPUTS = {
    "data/runs/morgana-v1/sft_train.jsonl": {
        "records": 50,
        "sha256": "277323c097305ebbee7bfa93cb27c34247800f2dec835f6d052ede7c2b178a7a",
    },
    "data/runs/morgana-v1/dev.jsonl": {
        "records": 10,
        "sha256": "cbce0b38bb6f8b8cbef0bc45fd52b5a8212a66445569dfe0a8c7e8e88f63ddc6",
    },
    "data/runs/morgana-v1/base_dev_outputs.jsonl": {
        "records": 10,
        "sha256": "840ce46d346cd01c9209174a8be1325415bd61c9da6567d21b85227f23dfd832",
    },
}
EXPECTED_FILE_HASHES = {
    "data/runs/morgana-v1/inputs/persona.json": "8b4be4ac72b0f90ff2bd875fe318319ce1498cd11cd503e39f782ca14b46ae90",
    "data/runs/morgana-v1/system_prompt.txt": "d2cbaaa6d603b66e123c0fc435bcb8477584ea487870087d1db74a8ae4a4938a",
}

def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def read_jsonl(path: Path) -> list[dict]:
    rows = []
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        assert line.strip(), f"{path}:{line_number} 空行"
        value = json.loads(line)
        assert isinstance(value, dict), f"{path}:{line_number} 不是对象"
        rows.append(value)
    return rows

validated_hashes = {}
loaded_inputs = {}
for relative_path, expected in EXPECTED_INPUTS.items():
    path = REPO_DIR / relative_path
    digest = sha256_file(path)
    rows = read_jsonl(path)
    assert digest == expected["sha256"], (relative_path, digest)
    assert len(rows) == expected["records"], (relative_path, len(rows))
    validated_hashes[relative_path] = digest
    loaded_inputs[relative_path] = rows
for relative_path, expected_digest in EXPECTED_FILE_HASHES.items():
    digest = sha256_file(REPO_DIR / relative_path)
    assert digest == expected_digest, (relative_path, digest)
    validated_hashes[relative_path] = digest

sft_rows = loaded_inputs["data/runs/morgana-v1/sft_train.jsonl"]
for index, row in enumerate(sft_rows):
    assert set(row) == {"messages"}, (index, row.keys())
    messages = row["messages"]
    assert isinstance(messages, list) and len(messages) >= 3, index
    assert messages[-1]["role"] == "assistant", index
    assert any(item.get("role") == "user" for item in messages), index
    for message in messages:
        assert set(message) == {"role", "content"}, (index, message)
        assert message["role"] in {"system", "user", "assistant"}, message
        assert isinstance(message["content"], str) and message["content"].strip(), message

dev_rows = loaded_inputs["data/runs/morgana-v1/dev.jsonl"]
assert len({row["id"] for row in dev_rows}) == 10
for row in dev_rows:
    assert set(row) == {"id", "scenario", "target_goals", "user"}, row.keys()
    assert all(isinstance(row[key], str) and row[key].strip() for key in ("id", "scenario", "user"))
    assert isinstance(row["target_goals"], list) and row["target_goals"]

base_rows = loaded_inputs["data/runs/morgana-v1/base_dev_outputs.jsonl"]
assert [row["user"] for row in base_rows] == [row["user"] for row in dev_rows]

train_config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
assert train_config["model"] == MODEL_ID, train_config["model"]
assert train_config["model_revision"] == MODEL_REVISION, train_config["model_revision"]
assert train_config["dataset"] == "data/runs/morgana-v1/sft_train.jsonl"
assert train_config["max_length"] == 1024
model_info = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION)
assert model_info.sha == MODEL_REVISION, model_info.sha
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
tokenizer = getattr(processor, "tokenizer", processor)
token_lengths = []
for index, row in enumerate(sft_rows):
    try:
        token_ids = processor.apply_chat_template(
            row["messages"], tokenize=True, add_generation_prompt=False, enable_thinking=False
        )
    except TypeError:
        token_ids = tokenizer.apply_chat_template(
            row["messages"], tokenize=True, add_generation_prompt=False
        )
    if isinstance(token_ids, dict):
        token_ids = token_ids["input_ids"]
    length = len(token_ids[0]) if token_ids and isinstance(token_ids[0], list) else len(token_ids)
    token_lengths.append(length)
    assert length <= 1024, f"SFT row {index} has {length} tokens"

data_validation = {
    "model": MODEL_ID,
    "model_revision": model_info.sha,
    "hashes": validated_hashes,
    "sft_records": len(sft_rows),
    "dev_records": len(dev_rows),
    "max_sft_tokens": max(token_lengths),
    "max_length": 1024,
}
validation_path = RUN_ROOT / "data_validation.json"
validation_path.write_text(
    json.dumps(data_validation, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
sync_to_drive(validation_path)
print(json.dumps(data_validation, ensure_ascii=False, indent=2))

In [ ]:
# 5. 训练/归档辅助函数。训练始终在独立子进程中执行并保留完整日志。
import math
import re

from peft import PeftConfig

TRAIN_ENV = {
    **os.environ,
    "CUDA_VISIBLE_DEVICES": "0",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "TOKENIZERS_PARALLELISM": "false",
}

def run_logged(command: list[str], log_path: Path) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w", encoding="utf-8") as log_file:
        log_file.write("COMMAND: " + subprocess.list2cmdline(command) + "\n")
        log_file.flush()
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            env=TRAIN_ENV,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

def find_final_adapter(output_dir: Path) -> Path:
    candidates = []
    for config_path in output_dir.rglob("adapter_config.json"):
        directory = config_path.parent
        if (directory / "adapter_model.safetensors").is_file() or (directory / "adapter_model.bin").is_file():
            match = re.search(r"checkpoint-(\d+)$", directory.name)
            step = int(match.group(1)) if match else -1
            candidates.append((step, config_path.stat().st_mtime_ns, directory))
    assert candidates, f"没有在 {output_dir} 找到可加载 adapter"
    adapter_dir = max(candidates)[2]
    PeftConfig.from_pretrained(adapter_dir)
    return adapter_dir

def read_logged_losses(output_dir: Path) -> list[float]:
    losses = []
    for path in output_dir.rglob("*.jsonl"):
        for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(row, dict) and isinstance(row.get("loss"), (int, float)):
                losses.append(float(row["loss"]))
    for path in output_dir.rglob("trainer_state.json"):
        state = json.loads(path.read_text(encoding="utf-8"))
        for row in state.get("log_history", []):
            if isinstance(row.get("loss"), (int, float)):
                losses.append(float(row["loss"]))
    return losses

shutil.copy2(CONFIG_PATH, RUN_ROOT / "training_config.yaml")
sync_to_drive(RUN_ROOT / "training_config.yaml")

In [ ]:
# 6. 用 YAML + max_steps=1 覆盖执行一个 optimizer step 冒烟训练。
SMOKE_DIR = RUN_ROOT / "smoke"
smoke_command = [
    "swift",
    "sft",
    str(CONFIG_PATH),
    "--max_steps",
    "1",
    "--save_strategy",
    "steps",
    "--save_steps",
    "1",
    "--output_dir",
    str(SMOKE_DIR),
    "--add_version",
    "false",
]
(RUN_ROOT / "smoke_command.json").write_text(
    json.dumps(smoke_command, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
try:
    run_logged(smoke_command, SMOKE_DIR / "console.log")
finally:
    if SMOKE_DIR.exists():
        sync_to_drive(SMOKE_DIR, "smoke")
    sync_to_drive(RUN_ROOT / "smoke_command.json")

smoke_losses = read_logged_losses(SMOKE_DIR)
assert smoke_losses, "冒烟训练没有记录 loss"
assert all(math.isfinite(loss) and loss > 0 for loss in smoke_losses), smoke_losses
SMOKE_ADAPTER = find_final_adapter(SMOKE_DIR)
print({"smoke_adapter": str(SMOKE_ADAPTER), "losses": smoke_losses})

In [ ]:
# 7. 写出仅供本次运行使用的 TransformersEngine 推理脚本。
INFER_SCRIPT = RUN_ROOT / "transformers_infer.py"
INFER_SCRIPT.write_text(
    r'''import argparse
import json
from pathlib import Path

import torch
from peft import PeftModel
from swift import InferRequest, RequestConfig, TransformersEngine, get_model_processor, get_template
from transformers import BitsAndBytesConfig

parser = argparse.ArgumentParser()
parser.add_argument("--adapter", required=True)
parser.add_argument("--input", required=True)
parser.add_argument("--output", required=True)
parser.add_argument("--max-tokens", type=int, required=True)
parser.add_argument("--temperature", type=float, required=True)
parser.add_argument("--top-p", type=float, required=True)
parser.add_argument("--top-k", type=int, required=True)
parser.add_argument("--repetition-penalty", type=float, required=True)
args = parser.parse_args()

model_id = "Qwen/Qwen3.5-2B"
model_revision = "965dcc54bc9c0591873df0e9869c056a54d323d1"
rows = [json.loads(line) for line in Path(args.input).read_text(encoding="utf-8").splitlines()]
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
model, processor = get_model_processor(
    model_id,
    revision=model_revision,
    torch_dtype=torch.float16,
    quantization_config=quantization_config,
    use_hf=True,
)
model = PeftModel.from_pretrained(model, args.adapter)
template = get_template(processor, enable_thinking=False)
engine = TransformersEngine(model, template=template)
requests = [InferRequest(messages=row["messages"]) for row in rows]
request_config = RequestConfig(
    max_tokens=args.max_tokens,
    temperature=args.temperature,
    top_p=args.top_p,
    top_k=args.top_k,
    repetition_penalty=args.repetition_penalty,
)
responses = engine.infer(requests, request_config=request_config)
assert len(responses) == len(rows), (len(responses), len(rows))
with Path(args.output).open("x", encoding="utf-8") as output_file:
    for index, response in enumerate(responses):
        choice = response.choices[0]
        record = {
            "index": index,
            "assistant": choice.message.content,
            "finish_reason": choice.finish_reason,
        }
        output_file.write(json.dumps(record, ensure_ascii=False) + "\n")
''',
    encoding="utf-8",
)

def run_transformers_inference(
    adapter: Path, input_path: Path, output_path: Path, log_path: Path, max_tokens: int
) -> None:
    command = [
        sys.executable,
        str(INFER_SCRIPT),
        "--adapter",
        str(adapter),
        "--input",
        str(input_path),
        "--output",
        str(output_path),
        "--max-tokens",
        str(max_tokens),
        "--temperature",
        "0.6",
        "--top-p",
        "0.8",
        "--top-k",
        "20",
        "--repetition-penalty",
        "1.45",
    ]
    run_logged(command, log_path)

In [ ]:
# 8. 从保存的冒烟 adapter 重载并生成一条非空回答。
system_prompt = (REPO_DIR / "data/runs/morgana-v1/system_prompt.txt").read_text(encoding="utf-8")
smoke_request_path = SMOKE_DIR / "reload_request.jsonl"
smoke_result_path = SMOKE_DIR / "reload_result.jsonl"
smoke_request = {
    "messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": dev_rows[0]["user"]},
    ]
}
smoke_request_path.write_text(
    json.dumps(smoke_request, ensure_ascii=False) + "\n", encoding="utf-8"
)
try:
    run_transformers_inference(
        SMOKE_ADAPTER, smoke_request_path, smoke_result_path, SMOKE_DIR / "reload_console.log", 64
    )
finally:
    sync_to_drive(SMOKE_DIR, "smoke")
smoke_result = read_jsonl(smoke_result_path)
assert len(smoke_result) == 1
assert isinstance(smoke_result[0]["assistant"], str) and smoke_result[0]["assistant"].strip()
print(smoke_result[0]["assistant"])

## 完整 SFT

下面的 `swift sft` 是新的操作系统子进程，不复用冒烟训练中的模型或 optimizer 状态。它只运行 YAML 中唯一的一组 1 epoch 主配置。

In [ ]:
# 9. 从干净进程运行完整 1 epoch，并定位最终 adapter/日志/state。
FULL_DIR = RUN_ROOT / "full"
full_command = [
    "swift",
    "sft",
    str(CONFIG_PATH),
    "--output_dir",
    str(FULL_DIR),
    "--add_version",
    "false",
]
(RUN_ROOT / "full_command.json").write_text(
    json.dumps(full_command, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
try:
    run_logged(full_command, FULL_DIR / "console.log")
finally:
    if FULL_DIR.exists():
        sync_to_drive(FULL_DIR, "full")
    sync_to_drive(RUN_ROOT / "full_command.json")

FULL_ADAPTER = find_final_adapter(FULL_DIR)
required_artifacts = {
    "args": list(FULL_DIR.rglob("args.json")),
    "trainer_state": list(FULL_DIR.rglob("trainer_state.json")),
    "logs": list(FULL_DIR.rglob("*.jsonl")),
}
assert all(required_artifacts.values()), required_artifacts
full_losses = read_logged_losses(FULL_DIR)
assert full_losses and all(math.isfinite(loss) for loss in full_losses), full_losses
assert any(loss > 0 for loss in full_losses), full_losses
(RUN_ROOT / "final_adapter.json").write_text(
    json.dumps({"path": str(FULL_ADAPTER)}, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
sync_to_drive(RUN_ROOT / "final_adapter.json")
print({"final_adapter": str(FULL_ADAPTER), "losses": full_losses})

In [ ]:
# 10. 将冻结 Dev 转成临时 messages 数据。
DEV_MESSAGES_PATH = RUN_ROOT / "dev_messages.jsonl"
with DEV_MESSAGES_PATH.open("x", encoding="utf-8") as output_file:
    for row in dev_rows:
        record = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": row["user"]},
            ]
        }
        output_file.write(json.dumps(record, ensure_ascii=False) + "\n")
assert len(read_jsonl(DEV_MESSAGES_PATH)) == 10
sync_to_drive(DEV_MESSAGES_PATH)
print(DEV_MESSAGES_PATH)

In [ ]:
# 11. Transformers backend 重载最终 LoRA，生成全部 10 条 Dev。
RAW_DEV_OUTPUTS = RUN_ROOT / "dev_outputs_raw.jsonl"
try:
    run_transformers_inference(
        FULL_ADAPTER,
        DEV_MESSAGES_PATH,
        RAW_DEV_OUTPUTS,
        RUN_ROOT / "dev_inference_console.log",
        256,
    )
finally:
    for partial_artifact in (RAW_DEV_OUTPUTS, RUN_ROOT / "dev_inference_console.log"):
        if partial_artifact.exists():
            sync_to_drive(partial_artifact)

raw_outputs = read_jsonl(RAW_DEV_OUTPUTS)
assert len(raw_outputs) == len(dev_rows) == 10
assert [row["index"] for row in raw_outputs] == list(range(10))
DEV_OUTPUTS_PATH = RUN_ROOT / "dev_outputs.jsonl"
with DEV_OUTPUTS_PATH.open("x", encoding="utf-8") as output_file:
    for source, generated in zip(dev_rows, raw_outputs, strict=True):
        answer = generated["assistant"]
        assert isinstance(answer, str) and answer.strip(), source["id"]
        record = {
            "id": source["id"],
            "scenario": source["scenario"],
            "target_goals": source["target_goals"],
            "user": source["user"],
            "assistant": answer,
            "finish_reason": generated["finish_reason"],
            "attempts": 1,
        }
        output_file.write(json.dumps(record, ensure_ascii=False) + "\n")
final_dev_rows = read_jsonl(DEV_OUTPUTS_PATH)
assert [row["user"] for row in final_dev_rows] == [row["user"] for row in dev_rows]
sync_to_drive(DEV_OUTPUTS_PATH)
print(f"generated {len(final_dev_rows)}/10 non-empty answers")

In [ ]:
# 12. 记录生成配置、后端差异和不静默重跑的机械检查摘要。
def has_repeated_span(text: str, span: int = 12, repeats: int = 3) -> bool:
    compact = re.sub(r"\s+", "", text)
    return any(compact.count(compact[index:index + span]) >= repeats for index in range(max(0, len(compact) - span + 1)))

format_issue_ids = []
truncated_ids = []
repetition_issue_ids = []
for source, generated in zip(dev_rows, final_dev_rows, strict=True):
    answer = generated["assistant"].strip()
    if not (answer.startswith("（") and "）" in answer):
        format_issue_ids.append(source["id"])
    if generated["finish_reason"] in {"length", "max_tokens"}:
        truncated_ids.append(source["id"])
    if has_repeated_span(answer):
        repetition_issue_ids.append(source["id"])

generation_metadata = {
    "schema_version": 1,
    "stage": "stage2_sft_dev",
    "repository_commit": repo_commit,
    "model": {"name": MODEL_ID, "revision": MODEL_REVISION},
    "adapter": str(FULL_ADAPTER.relative_to(RUN_ROOT)),
    "backend": "ms-swift TransformersEngine",
    "generation": {
        "max_new_tokens": 256,
        "request_config_mapping": {"max_new_tokens": "max_tokens"},
        "temperature": 0.6,
        "top_p": 0.8,
        "top_k": 20,
        "repetition_penalty": 1.45,
        "enable_thinking": False,
    },
    "backend_differences_from_base": {
        "presence_penalty": "Base 使用 0.4；TransformersEngine 不支持等价 presence_penalty，故未应用。",
        "presence_context_size": "随 presence_penalty 一并不适用。",
    },
    "inputs": {
        "dev_sha256": validated_hashes["data/runs/morgana-v1/dev.jsonl"],
        "records": 10,
    },
    "outputs": {
        "file": DEV_OUTPUTS_PATH.name,
        "sha256": sha256_file(DEV_OUTPUTS_PATH),
        "records": len(final_dev_rows),
    },
}
metadata_path = RUN_ROOT / "dev_generation_meta.json"
metadata_path.write_text(
    json.dumps(generation_metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)

run_summary = {
    "run_id": RUN_ID,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "smoke": {"optimizer_steps": 1, "losses": smoke_losses, "reload_nonempty": True},
    "full_train": {
        "epochs": 1,
        "losses": full_losses,
        "adapter": str(FULL_ADAPTER.relative_to(RUN_ROOT)),
    },
    "dev": {
        "records": len(final_dev_rows),
        "all_nonempty": all(row["assistant"].strip() for row in final_dev_rows),
        "format_issue_ids": format_issue_ids,
        "truncated_ids": truncated_ids,
        "repetition_issue_ids": repetition_issue_ids,
    },
    "policy": "机械检查异常只记录，不静默修改超参数或重跑。",
}
summary_path = RUN_ROOT / "stage2_summary.json"
summary_path.write_text(
    json.dumps(run_summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
notes_path = RUN_ROOT / "stage2_notes.md"
notes_path.write_text(
    "# Stage 2 SFT 运行摘要\n\n"
    f"- Run ID：`{RUN_ID}`\n"
    f"- Repository commit：`{repo_commit}`\n"
    f"- Model revision：`{MODEL_REVISION}`\n"
    f"- Dev：{len(final_dev_rows)}/10 非空\n"
    f"- 格式异常：{format_issue_ids or '无'}\n"
    f"- 截断：{truncated_ids or '无'}\n"
    f"- 复读嫌疑：{repetition_issue_ids or '无'}\n"
    "- 后端差异：TransformersEngine 不支持 Base 的 presence_penalty=0.4。\n"
    "- 本次不因机械检查异常静默修改超参数或重跑。\n",
    encoding="utf-8",
)
for artifact in (metadata_path, summary_path, notes_path):
    sync_to_drive(artifact)
print(json.dumps(run_summary, ensure_ascii=False, indent=2))

In [ ]:
# 13. 最终归档。run-id 目录在创建时要求不存在，因此不会覆盖旧尝试。
sync_to_drive(RUN_ROOT, ".")
required_drive_artifacts = [
    DRIVE_RUN_ROOT / "environment/environment.json",
    DRIVE_RUN_ROOT / "smoke",
    DRIVE_RUN_ROOT / "full",
    DRIVE_RUN_ROOT / "dev_outputs.jsonl",
    DRIVE_RUN_ROOT / "dev_generation_meta.json",
    DRIVE_RUN_ROOT / "stage2_summary.json",
    DRIVE_RUN_ROOT / "stage2_notes.md",
]
assert all(path.exists() for path in required_drive_artifacts), required_drive_artifacts
print(f"Stage 2 已归档：{DRIVE_RUN_ROOT}")
print("请依据 stage2_summary.json 与 Base 输出完成定性复盘，再更新仓库 RUNLOG.md。")